In [1]:
import argparse
import torch
import os
import json
from pilgrim import Trainer, Pilgrim
from pilgrim import count_parameters, generate_inverse_moves, load_permutation_data

In [2]:
# Set the arguments directly in the notebook
args = argparse.Namespace(
    epochs=500,  # Number of training epochs
    batch_size=10000,  # Batch size
    lr=0.001,  # Learning rate
    dropout=0.0,  # Dropout
    optimizer="Adam",  # Optimizer (Adam or AdamSF)
    activation="relu",  # Activation function (relu or mish)
    use_batch_norm=True,  # Batch normalization usage (True or False, default True)
    K_min=1,  # Minimum K value for random walks
    K_max=50,  # Maximum K value for random walks
    weights='',  # Path to file with model weights
    device_id=0,  # Device ID
    alpha=1,  # TD-learning parameter, avg 1/α steps
    permutation = 'standard',
    permutation_size=35,  # Cube size (e.g., 4 for 4x4x4 cube)
    permutation_type="all",  # Cube type (qtm or all)
    hd1=2000,  # Size of the first hidden layer
    hd2=1000,  # Size of the second hidden layer (0 means no second layer)
    nrd=2  # Number of residual blocks (0 means no residual blocks)
)

In [3]:
# Set device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu", args.device_id)
print(f"Start training with {device}.")

Start training with cuda:0.


In [4]:
# Load cube data (moves and names)
all_moves, move_names = load_permutation_data(args.permutation, args.permutation_size, args.permutation_type, device)

File saved to generators/all_standard35.json


In [5]:
# Derive important cube parameters from the loaded data
n_gens = all_moves.size(0)  # Number of moves
state_size = all_moves.size(1)  # Size of the state representation
face_size = state_size // 6  # Size of one face of the cube

# Generate inverse moves
inverse_moves = torch.tensor(generate_inverse_moves(move_names), dtype=torch.int64, device=device)
if args.permutation == 'standard':
    V0 = torch.arange(args.permutation_size, dtype=torch.int8, device=device)
elif args.permutation == 'cube':
    V0 = torch.arange(6, dtype=torch.int8, device=device).repeat_interleave(face_size)
else:
    raise ValueError(f"Invalid permutation: {args.permutation}")

# Infer model mode based on hd1, hd2, and nrd
if args.hd2 == 0 and args.nrd == 0:
    mode = "MLP1"
elif args.hd2 > 0 and args.nrd == 0:
    mode = "MLP2"
elif args.hd2 > 0 and args.nrd > 0:
    mode = "MLP2RB"
else:
    raise ValueError("Invalid combination of hd1, hd2, and nrd.")

In [6]:
# Initialize the Pilgrim model
model = Pilgrim(
    state_size=state_size,
    hd1=args.hd1,
    hd2=args.hd2,
    nrd=args.nrd,
    dropout_rate=args.dropout,
    activation_function=args.activation
).to(device)

if len(args.weights) > 0:
    model.load_state_dict(torch.load(f"weights/{args.weights}.pth", weights_only=True, map_location=device))
    print(f"Weights {args.weights} loaded.")

# Calculate the number of model parameters
num_parameters = count_parameters(model)
param_million = round(num_parameters / 1_000_000)  # Get the number of parameters in millions

# Create the training name based on mode, hidden layers, residual blocks, and number of parameters
name = f"{args.permutation}_{args.permutation_size}_{args.permutation_type}_{mode}_{param_million:02d}M"

name

'standard_35_all_MLP2RB_06M'

In [7]:
# Create the trainer
trainer = Trainer(
    net=model,
    num_epochs=args.epochs,
    device=device,
    batch_size=args.batch_size,
    lr=args.lr,
    name=name,
    K_min=args.K_min,
    K_max=args.K_max,
    all_moves=all_moves,
    inverse_moves=inverse_moves,
    V0=V0,
    α=args.alpha
)

# Save the arguments to a log file
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)
args_dict = vars(args)  # Convert the args object to a dictionary
args_dict["model_name"] = name
args_dict["model_mode"] = mode
args_dict["model_id"] = trainer.id
args_dict["num_parameters"] = num_parameters

# Save the args and model information in JSON format
with open(f"{log_dir}/model_{name}_{trainer.id}.json", "w") as f:
    json.dump(args_dict, f, indent=4)

In [8]:
# Start the training process
trainer.run()

[2024-12-22 21:55:25] Saved weights at epoch     1. Train Loss: 173.33
[2024-12-22 21:55:30] Saved weights at epoch     2. Train Loss: 113.27
[2024-12-22 21:55:41] Saved weights at epoch     4. Train Loss: 93.60
[2024-12-22 21:56:04] Saved weights at epoch     8. Train Loss: 82.63
[2024-12-22 21:56:48] Saved weights at epoch    16. Train Loss: 74.93
[2024-12-22 21:58:15] Saved weights at epoch    32. Train Loss: 70.68
[2024-12-22 22:01:13] Saved weights at epoch    64. Train Loss: 67.70
[2024-12-22 22:07:02] Saved weights at epoch   128. Train Loss: 64.71
[2024-12-22 22:18:49] Saved weights at epoch   256. Train Loss: 63.57
[2024-12-22 22:41:37] Finished. Saved final weights at epoch 500. Train Loss: 62.73.


In [9]:
############## Test the model ###############

In [13]:
import numpy as np
import argparse
import torch
import os
import json
import time
from pilgrim import Pilgrim, Searcher
from pilgrim import count_parameters, generate_inverse_moves, load_permutation_data

def scramble_given_state(list_generators, n_scrambles,    state_to_scramble  = '01234...'   ):
    if state_to_scramble == '01234...':
        state_size = len( list_generators[0] )
        state_to_scramble = np.arange(state_size)  
    state_current = state_to_scramble
    if isinstance(state_current,list): 
        state_current = np.asarray(state_current)
    elif isinstance(state_current,range): 
        state_current = np.asarray(state_current)
    n_gens = len(list_generators)
    for k in range(n_scrambles):
        IX_move = np.random.randint(0, n_gens, dtype = int) # random moves indixes
        state_current = state_current[ list_generators[IX_move]] # all_moves[IX_moves,:] ] 
    return state_current

In [14]:
args = argparse.Namespace(
    permutation = 'standard',
    permutation_size=35,  # Cube size (e.g., 4 for 4x4x4 cube)
    permutation_type="all",  # Cube type (qtm or all)
    tests="",
    weights="weights/standard_35_all_MLP2RB_06M_1734900918_e00500final.pth",
    B=2**18,
    num_attempts=2,
    num_steps=200,
    tests_num=1,
    device_id=0,
    verbose=0,
    shift=0,
    return_tree=0
)

log_dir = "logs"
forest_dir = "forest"

# Load model info
with open(f"{log_dir}/model_{'_'.join(args.weights.split('/')[-1].split('_')[:-1])}.json", "r") as json_file:
    info = json.load(json_file)

In [15]:
# Load cube data (moves and names)
all_moves, move_names = load_permutation_data(args.permutation, args.permutation_size, args.permutation_type, device)

# Derive important cube parameters from the loaded data
n_gens = all_moves.size(0)  # Number of moves
state_size = all_moves.size(1)  # Size of the state representation
face_size = state_size // 6  # Size of one face of the cube

# Generate inverse moves
inverse_moves = torch.tensor(generate_inverse_moves(move_names), dtype=torch.int64, device=device)
if args.permutation == 'standard':
    V0 = torch.arange(args.permutation_size, dtype=torch.int8, device=device)
elif args.permutation == 'cube':
    V0 = torch.arange(6, dtype=torch.int8, device=device).repeat_interleave(face_size)
else:
    raise ValueError(f"Invalid permutation: {args.permutation}")

In [16]:
# Load model and weights
model = Pilgrim(state_size=state_size, 
                hd1=info['hd1'], hd2=info['hd2'], nrd=info['nrd'], 
                activation_function=info.get('activation', 'relu'), 
                use_batch_norm=info.get('use_batch_norm', True))
model.load_state_dict(torch.load(args.weights, weights_only=False, map_location=device))
model.eval()

Pilgrim(
  (input_layer): Linear(in_features=35, out_features=2000, bias=True)
  (bn1): BatchNorm1d(2000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (activation): ReLU()
  (dropout): Dropout(p=0.1, inplace=False)
  (hidden_layer): Linear(in_features=2000, out_features=1000, bias=True)
  (bn2): BatchNorm1d(1000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (residual_blocks): ModuleList(
    (0-1): 2 x ResidualBlock(
      (fc1): Linear(in_features=1000, out_features=1000, bias=True)
      (bn1): BatchNorm1d(1000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activation): ReLU()
      (dropout): Dropout(p=0.1, inplace=False)
      (fc2): Linear(in_features=1000, out_features=1000, bias=True)
      (bn2): BatchNorm1d(1000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (output_layer): Linear(in_features=1000, out_features=1, bias=True)
)

In [23]:
state_start = V0
n_scrambles_starting_state = 1000
state_start = scramble_given_state( all_moves, n_scrambles_starting_state, state_start ) 
tests = [state_start]

In [24]:
# Initialize Searcher object
searcher = Searcher(model=model, all_moves=all_moves, V0=V0, device=device, verbose=args.verbose)

# Extract epoch information from weights file name
epoch = args.weights.split('_')[-1][:-4]

# Prepare log file
os.makedirs(log_dir, exist_ok=True)
log_file_add = ""
if len(args.tests) > 0:
    log_file_add = log_file_add + "tests_" + args.tests.split("/")[1].split(".")[0]
if args.shift > 0:
    log_file_add = log_file_add + f"_shift{args.shift}"
if len(log_file_add) > 0:
    log_file_add = "_" + log_file_add
log_file = f"{log_dir}/test_{info['model_name']}_{info['model_id']}_{epoch}_B{args.B}{log_file_add}.json"

In [25]:
results = []
total_length = 0
t1 = time.time()
for i, state in enumerate(tests, start=0):
    solution_time_start = time.time()
    result = searcher.get_solution(
        state, B=args.B, 
        num_steps=args.num_steps, num_attempts=args.num_attempts, 
        return_tree=args.return_tree
    )
    moves, attempts = result[:2]
    if args.return_tree and moves is not None:
        tree = result[2]
        os.makedirs(forest_dir, exist_ok=True)
        torch.save(tree.cpu(), f"{forest_dir}/tree_{args.permutation}_{args.permutation_size}_{args.permutation_type}_i{i+args.shift:04d}_B{args.B:08d}_{info['model_id']}.pt")  
        torch.save(state.cpu(), f"{forest_dir}/state_{args.permutation}_{args.permutation_size}_{args.permutation_type}_i{i+args.shift:04d}_B{args.B:08d}_{info['model_id']}.pt") 

    solution_time_end = time.time()
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())
    
    if moves is not None:
        solution_length = len(moves)
        total_length += solution_length
        
        result = {
            "test_num": i+args.shift,
            "solution_length": solution_length,
            "attempts": attempts + 1,
            "time": round(solution_time_end - solution_time_start, 2),
            "moves": moves.tolist()
        }
        
        # Print solution length for each solved cube
        print(f"[{timestamp}] Solution {i}: Length = {solution_length}")
    else:
        # If no solution is found
        result = {
            "test_num": i+args.shift,
            "solution_length": None,
            "attempts": None,
            "time": round(solution_time_end - solution_time_start, 2),
            "moves": None
        }
        print(f"[{timestamp}] Solution {i} not found")
    
    results.append(result)

    # Append new result to the log file
    with open(log_file, 'w') as f:
        json.dump(results, f, indent=4)

t2 = time.time()

# Calculate average solution length
solved_results = [r for r in results if r["solution_length"] is not None]
avg_length = total_length / len(solved_results) if solved_results else 0

# Print completion message with average solution length
print(f"Test completed in {(t2 - t1):.2f}s.")
print(f"Average solution length: {avg_length:.2f}.")
print(f"Solved {len(solved_results)}/{args.tests_num} permutations.")
print(f"Results saved to {log_file}.")

[2024-12-22 22:53:52] Solution 0 not found
Test completed in 262.04s.
Average solution length: 0.00.
Solved 0/1 permutations.
Results saved to logs/test_standard_35_all_MLP2RB_06M_1734900918_e00500final_B262144.json.
